In [0]:
# ============================================================================
# INSTALL REQUIRED LIBRARY (run this cell first)
# ============================================================================
%pip install databricks-vectorsearch --quiet
dbutils.library.restartPython()

In [0]:
# ============================================================================
# VECTOR SEARCH SETUP - Agentic RAG with Databricks Vector Search
# ============================================================================
from databricks.vector_search.client import VectorSearchClient
from openai import OpenAI
import pandas as pd

print("="*70)
print("🚀 SETTING UP VECTOR SEARCH FOR AGENTIC RAG")
print("="*70)

# Initialize clients
OPENAI_API_KEY = dbutils.secrets.get(scope="openai", key="GDPR_agent")
openai_client = OpenAI(api_key=OPENAI_API_KEY)
vsc = VectorSearchClient()

# ============================================================================
# STEP 1: Create Vector Search Endpoint (run once)
# ============================================================================
ENDPOINT_NAME = "gdpr_rag_endpoint"

try:
    endpoint = vsc.get_endpoint(ENDPOINT_NAME)
    print(f"✓ Endpoint '{ENDPOINT_NAME}' already exists")
except Exception:
    print(f"Creating endpoint '{ENDPOINT_NAME}'...")
    vsc.create_endpoint(
        name=ENDPOINT_NAME,
        endpoint_type="STANDARD"  # Use STANDARD for production
    )
    print(f"✓ Endpoint created")


In [0]:
%sql
-- ============================================================================
-- ENABLE CHANGE DATA FEED ON SOURCE TABLES (run this first)
-- ============================================================================

-- Enable CDF on GDPR fines table
ALTER TABLE main.default.gdpr_chunk_embeddings 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Enable CDF on GDPR law table
ALTER TABLE main.default.gdpr_statutory_legislation_embeddings 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Enable CDF on privacy policy table
ALTER TABLE main.default.retail_corporate_policy_embeddings 
SET TBLPROPERTIES (delta.enableChangeDataFeed = true);

-- Verify CDF is enabled
DESCRIBE DETAIL main.default.gdpr_chunk_embeddings;
DESCRIBE DETAIL main.default.gdpr_statutory_legislation_embeddings;
DESCRIBE DETAIL main.default.retail_corporate_policy_embeddings;

In [0]:
indices_config = [
    {
        "index_name": "main.default.gdpr_fines_vector_index",
        "source_table": "main.default.gdpr_chunk_embeddings",
        "embedding_col": "embedding",
        "text_col": "translated_text",
        "primary_key": "chunk_id", 
        "description": "GDPR Historical Fines enforcement tracker"
    },
    {
        "index_name": "main.default.gdpr_law_vector_index",
        "source_table": "main.default.gdpr_statutory_legislation_embeddings",
        "embedding_col": "embedding",
        "text_col": "text_content",
        "primary_key": "chunk_id", 
        "description": "GDPR Chapter 3 statutory legislation"
    },
    {
        "index_name": "main.default.privacy_policy_vector_index",
        "source_table": "main.default.retail_corporate_policy_embeddings",
        "embedding_col": "embedding",
        "text_col": "text_content",
        "primary_key": "chunk_id",  
        "description": "Corporate privacy policy"
    }
]

In [0]:
# ============================================================================
# RECREATE INDICES WITH CORRECT PRIMARY KEY
# ============================================================================
import time

print("="*70)
print("📊 CREATING VECTOR SEARCH INDICES")
print("="*70)

for config in indices_config:
    print(f"\n🔨 Processing: {config['index_name']}")
    print(f"   Source: {config['source_table']}")
    print(f"   Primary Key: {config['primary_key']}")
    
    # Step 1: Delete if exists
    try:
        index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=config["index_name"])
        status = index.describe().get('status', {}).get('detailed_state', 'UNKNOWN')
        print(f"   ⚠️  Index exists (Status: {status}) - deleting to recreate...")
        vsc.delete_index(endpoint_name=ENDPOINT_NAME, index_name=config["index_name"])
        print(f"   ✓ Delete initiated")
        
        # ✅ Wait for deletion to complete
        print(f"   ⏳ Waiting for deletion to complete...")
        for i in range(30):  # Wait up to 60 seconds
            time.sleep(2)
            try:
                vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=config["index_name"])
                print(f"   ... still deleting (attempt {i+1}/30)")
            except Exception:
                print(f"   ✓ Deletion complete!")
                break
        else:
            print(f"   ⚠️  Deletion taking longer than expected, but continuing...")
            time.sleep(5)  # Extra buffer
            
    except Exception:
        print(f"   Index doesn't exist yet")

    # Step 2: Create index
    print(f"   Creating new index...")
    try:
        if config["index_name"] == "main.default.gdpr_fines_vector_index":
            print(f"   🔍 Enabling HYBRID search (BM25 + Dense)")
            index = vsc.create_delta_sync_index(
                endpoint_name=ENDPOINT_NAME,
                index_name=config["index_name"],
                source_table_name=config["source_table"],
                pipeline_type="TRIGGERED",
                primary_key=config["primary_key"],
                embedding_dimension=1536,
                embedding_vector_column=config["embedding_col"],
                columns_to_sync=["source_file_name", "translated_text", "chunk_text"]
            )
        else:
            index = vsc.create_delta_sync_index(
                endpoint_name=ENDPOINT_NAME,
                index_name=config["index_name"],
                source_table_name=config["source_table"],
                pipeline_type="TRIGGERED",
                primary_key=config["primary_key"],
                embedding_dimension=1536,
                embedding_vector_column=config["embedding_col"]
            )
        print(f"   ✅ Index created successfully")
        print(f"   ⏳ Index is syncing... this will take 2-5 minutes")
    except Exception as create_error:
        print(f"   ❌ Error creating index: {create_error}")

print("\n" + "="*70)
print("✅ INDEX CREATION COMPLETE")
print("="*70)
print("\n⏳ Wait 2-5 minutes, then run Cell 6 to check status")

In [0]:
# ============================================================================
# CHECK INDEX STATUS
# ============================================================================

print("="*70)
print("📊 CHECKING VECTOR SEARCH INDEX STATUS")
print("="*70)

index_names = [
    "main.default.gdpr_fines_vector_index",
    "main.default.gdpr_law_vector_index",
    "main.default.privacy_policy_vector_index"
]

for index_name in index_names:
    print(f"\n🔍 Index: {index_name}")
    print("-" * 70)
    
    try:
        index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=index_name)
        status_info = index.describe()
        
        # Extract key status fields
        detailed_state = status_info.get('status', {}).get('detailed_state', 'UNKNOWN')
        ready = status_info.get('status', {}).get('ready', False)
        message = status_info.get('status', {}).get('message', 'No message')
        
        # Get delta sync status - try multiple possible paths
        delta_sync = status_info.get('delta_sync_index_spec', {})
        source_table = delta_sync.get('source_table', 'N/A')
        
        # Try different possible field names for primary key
        primary_key = (
            delta_sync.get('primary_key') or 
            delta_sync.get('primary_key_columns') or
            status_info.get('primary_key') or
            'N/A'
        )
        
        print(f"   Status: {detailed_state}")
        print(f"   Ready: {'✅ Yes' if ready else '❌ No'}")
        print(f"   Source Table: {source_table}")
        print(f"   Primary Key: {primary_key}")
        print(f"   Message: {message}")
        
        # Check if ONLINE
        if 'ONLINE' in detailed_state:
            print(f"   🎉 Index is ready for searching!")
        elif 'PROVISIONING' in detailed_state or 'SYNCING' in detailed_state:
            print(f"   ⏳ Index is still syncing... wait a few minutes")
        elif 'FAILED' in detailed_state:
            print(f"   ❌ Index has failed - may need to recreate")
        else:
            print(f"   ⚠️  Index may have issues")
            
    except Exception as e:
        print(f"   ❌ Error: {e}")
        print(f"   Index may not exist yet")

print("\n" + "="*70)
print("✅ STATUS CHECK COMPLETE")
print("="*70)
print("\nℹ️  Indices are ready when status contains 'ONLINE'")